In [6]:
library(dplyr)
library(rtracklayer)

library(data.table)
library(stringr)


Attaching package: 'data.table'


The following object is masked from 'package:GenomicRanges':

    shift


The following object is masked from 'package:IRanges':

    shift


The following objects are masked from 'package:S4Vectors':

    first, second


The following objects are masked from 'package:dplyr':

    between, first, last




In [2]:
safe_median <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) NA_real_ else median(x)
}

safe_max <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) NA_real_ else max(x)
}

safe_min <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) NA_real_ else min(x)
}

In [3]:
# gtf <- import("/new-stg/home/alex/ref/genome/hg38/annotations/gencode.v48.annotation.gtf")

# genes <- as.data.frame(gtf) %>%
#   filter(type == "gene") %>%
#   transmute(
#     seqnames = as.character(seqnames),
#     start = start,
#     end = end,
#     strand = as.character(strand),
#     gene_id = gene_id,
#     gene_name = gene_name,
#     gene_type = gene_type,
#     gene_length = end - start + 1
#   )

# genes %>%
#   filter(gene_name == "IRF1")

In [4]:
# gtf_df <- as.data.frame(gtf)

# tx_lengths_pc1 <- gtf_df %>%
#   filter(
#     type == "transcript",
#     gene_type == "protein_coding",
#     transcript_type == "protein_coding"
#   ) %>%
#   mutate(
#     transcript_span = end - start + 1,
#     is_ccds  = !is.na(tag) & grepl("CCDS", tag),
#     is_mane  = !is.na(tag) & grepl("MANE_Select", tag),
#     is_basic = !is.na(tag) & grepl("basic", tag)
#   ) %>%
#   group_by(gene_name, gene_id) %>%
#   summarize(
#     n_pc_tx = n(),

#     min_pc_tx_span    = safe_min(transcript_span),
#     median_pc_tx_span = safe_median(transcript_span),
#     max_pc_tx_span    = safe_max(transcript_span),

#     median_ccds_pc_tx_span = safe_median(transcript_span[is_ccds]),
#     max_ccds_pc_tx_span    = safe_max(transcript_span[is_ccds]),

#     median_basic_pc_tx_span = safe_median(transcript_span[is_basic]),
#     max_basic_pc_tx_span    = safe_max(transcript_span[is_basic]),

#     mane_tx_span = safe_median(transcript_span[is_mane]),

#     .groups = "drop"
#   ) %>%
#   mutate(
#     length_for_delay_model = coalesce(
#       mane_tx_span,
#       median_ccds_pc_tx_span,
#       median_basic_pc_tx_span,
#       median_pc_tx_span
#     ),
#     log10_length_for_delay_model = log10(length_for_delay_model)
#   )

In [5]:
# tx_lengths_pc1 %>% filter(gene_name == "HDAC9") %>% select(length_for_delay_model)

In [ ]:
# saveRDS(tx_lengths_pc, '../intmd/tx_lengths.rds')

In [7]:
gtf_path <- "/new-stg/home/alex/ref/genome/hg38/annotations/gencode.v48.annotation.gtf"

# Read transcript rows while retaining the complete attribute column
tx_raw <- fread(
  cmd = sprintf(
    "awk -F '\\t' '$3 == \"transcript\"' %s",
    shQuote(gtf_path)
  ),
  sep = "\t",
  header = FALSE,
  quote = "",
  select = c(4, 5, 9),
  col.names = c("start", "end", "attributes"),
  showProgress = FALSE
)

# Extract identifiers and detect every repeated tag directly
tx_tag_lookup <- tx_raw %>%
  transmute(
    transcript_id = str_match(
      attributes,
      'transcript_id "([^"]+)"'
    )[, 2],

    is_mane = str_detect(
      attributes,
      'tag "MANE_Select"'
    ),

    is_ccds = str_detect(
      attributes,
      'tag "CCDS"'
    ),

    is_basic = str_detect(
      attributes,
      'tag "basic"'
    )
  )

In [9]:
gtf <- import("/new-stg/home/alex/ref/genome/hg38/annotations/gencode.v48.annotation.gtf")
gtf_df <- as.data.frame(gtf)

In [10]:
tx_lengths_pc <- gtf_df %>%
  filter(
    type == "transcript",
    gene_type == "protein_coding",
    transcript_type == "protein_coding"
  ) %>%
  left_join(tx_tag_lookup, by = "transcript_id") %>%
  mutate(
    transcript_span = end - start + 1,
    is_mane  = coalesce(is_mane, FALSE),
    is_ccds  = coalesce(is_ccds, FALSE),
    is_basic = coalesce(is_basic, FALSE)
  ) %>%
  group_by(gene_name, gene_id) %>%
  summarize(
    n_pc_tx = n(),

    min_pc_tx_span    = safe_min(transcript_span),
    median_pc_tx_span = safe_median(transcript_span),
    max_pc_tx_span    = safe_max(transcript_span),

    median_ccds_pc_tx_span = safe_median(transcript_span[is_ccds]),
    max_ccds_pc_tx_span    = safe_max(transcript_span[is_ccds]),

    median_basic_pc_tx_span = safe_median(transcript_span[is_basic]),
    max_basic_pc_tx_span    = safe_max(transcript_span[is_basic]),

    mane_tx_span = safe_median(transcript_span[is_mane]),

    .groups = "drop"
  ) %>%
  mutate(
    length_for_delay_model = coalesce(
      mane_tx_span,
      median_ccds_pc_tx_span,
      median_basic_pc_tx_span,
      median_pc_tx_span
    ),
    log10_length_for_delay_model =
      log10(length_for_delay_model)
  )

In [11]:
# Number and proportion of genes for which MANE supplied the final length
tx_lengths_pc %>%
  summarize(
    total_genes = n(),
    n_MANE_used = sum(!is.na(mane_tx_span)),
    prop_MANE_used = mean(!is.na(mane_tx_span)),
    percent_MANE_used = 100 * mean(!is.na(mane_tx_span))
  )

total_genes,n_MANE_used,prop_MANE_used,percent_MANE_used
<int>,<int>,<dbl>,<dbl>
19746,19226,0.9736656,97.36656


In [14]:
query.gene = 'NFKBIA'
tx_lengths_pc %>% filter(gene_name == query.gene) %>% select(length_for_delay_model)

length_for_delay_model
<dbl>
3237


In [16]:
gtf_df %>%
  filter(
    type == "transcript",
    gene_type == "protein_coding",
    transcript_type == "protein_coding"
  ) %>%
  left_join(tx_tag_lookup, by = "transcript_id") %>%
  filter(is_mane == TRUE) %>%
  count(gene_name, gene_id) %>%
  count(n)

Storing counts in `nn`, as `n` already present in input
i Use `name = "new_name"` to pick a new name.


n,nn
<int>,<int>
1,19226


In [18]:
saveRDS(tx_lengths_pc, '../intmd/tx_lengths.rds')